# HRAI API demo notebook

## Spuštění API a notebooku současně

1. V jednom terminálu spusťte API server:
   ```bash
   uvicorn main:app
   ```
   Server poběží na `http://127.0.0.1:8000`.

2. V dalším terminálu spusťte Jupyter notebook v prohlížeči (ne v IDE — může dojít ke konfliktu portů):
   ```bash
   jupyter notebook --no-browser
   ```
   Poté notebook otevřete v prohlížeči na URL, kterou Jupyter vypíše.

3. Spouštějte buňky postupně.

---

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

In [1]:
import os, json, random, textwrap, requests
from pathlib import Path

from pos_extraction import text_to_ngrams
from query import extract_from_resume
from dotenv import load_dotenv

from rich import print as rprint

In [ ]:
# pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN:
os.environ['HF_TOKEN'] = ''

In [2]:
load_dotenv()

BASE_URL = 'http://127.0.0.1:8000'

## Ukázky

In [3]:
resumes = json.loads(Path(os.path.join('data', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)

print(textwrap.shorten(text, width=600, placeholder='...'))

vybavený vynikajícími vyjednávacími schopnostmi, přehledem o trhu a obchodní prozíravostí nezbytnou k tomu, aby vedl bojující a nově založené společnosti k finančnímu úspěchu. ❖ Organizační schopnosti ❖ Marketing, generování potenciálních zákazníků ❖ Strategický a poradenský prodej ❖ Interpersonální dovednosti a dovednosti v oblasti spolupráce ❖ Prodej B do B ❖ Udržení zákazníků ❖ Správa účtů ❖ Rozvoj/rozšiřování teritorií ❖ C-Level a technické prezentace ❖ Projektový management Vynikající prodejní techniky Úspěchy Členem od roku, zvolen do výkonné rady v roce . Vzal si volno, aby pomohl s...


#### vytvoření frází

In [4]:
ngrams = text_to_ngrams(text)
print(', '.join(sorted(ngrams)))

#1 v doporučeních, 000 zaměstnanců, 1 milion, 1 milion obchodníků, 1 provize, 10 regionu, 10 regionu experience, 100 cílů, 100 cílů proniknutí, 100 kvóty, 100 prodejních kvót, 105 tisíc dolarů, 109 škol, 109 škol jazyky, 11 měsíců, 114 týmů, 12 za čtvrtletí, 14 000 zaměstnanců, 2 b, 2 b lead, 2 místo, 200 zástupců, 200 účty, 25 místo, 3 kontejnery, 3 místě, 3 měsíce, 30 bankami, 30 doporučení, 30 účtů, 300 lidí, 35 států, 50 tisíc usd, 53 prodejů, 6 místo, 6 místě, 600 poboček, 7 měsíců, 7 měsíců mluvčí, 8 výplatních období, account executive klienti, account management, account management b, account manager, account manager účty, achievement award, akce, akce ve spolupráci, akcí, akcích, akcích 300, akcích 300 lidí, aktivací 68, b 2, b 2 b, b b, b b 2, b do b, b lead, b lead generation, b to b, b udržení, b udržení zákazníků, bakaláři, bakaláři umění, balíčky, balíčky za leden, bankami, banky, banky uzavřely, banky uzavřely partnerství, barové sady, basketbalové ligy, basketbalové lig

#### získání klíčových slov a oborů (pomocí pick_resume)

In [5]:
entities = extract_from_resume(text)
skills = [e for e in entities if e.entity_type == 'skill']
occupations = [e for e in entities if e.entity_type == 'occupation']

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [6]:
print(f"dovednosti - {len(skills)}:")
for s in skills:
    print(f'{s.label}\n'
          f'skore: {round(s.cosine_score,5)}\n'
          f'{s.description}\n')

dovednosti - 30:
spravovat účty
skore: 0.9565
Provádět správu účetnictví a finanční činnosti organizace, dohlížet na správné vedení všech dokumentů, správnost všech informací a výpočtů a přijímat řádná rozhodnutí.

převzít odpovědnost
skore: 0.91701
Přijmout zodpovědnost a odpovědnost za vlastní profesní rozhodnutí a činy, jakož i za ty, které byly delegovány na jiné.

správa kanceláře
skore: 0.90225
Administrativní procesy související se správními činnostmi kancelářského prostředí. Tyto činnosti nebo postupy mohou zahrnovat finanční plánování, vedení záznamů a fakturaci a řízení obecné logistiky organizace.

podnikání
skore: 0.89804
Rozvoj, organizace a správa vlastního podniku

organizační struktura
skore: 0.86477
Systém jednotlivých oddělení v organizaci, jakož i příslušní pracovníci, jejich úlohy a odpovědnosti.

poskytovat vynikající služby
skore: 0.85705
Poskytovat prvotřídní zákaznické služby a překračovat očekávání zákazníků; vybudovat si pověst výjimečného poskytovatele služeb

In [7]:
print(f"\nzaměstnání - {len(occupations)}:")
for o in occupations:
     print(f'název: {o.label}\n'
          f'skore: {round(o.cosine_score,5)}\n'
          f'{o.description}\n')


zaměstnání - 4:
název: mluvčí
skore: 1.0
Mluvčí hovoří jménem společností nebo organizací. Využívají komunikační strategie k zastupování klientů prostřednictvím veřejných oznámení a konferencí. Vykreslují své klienty v pozitivním světle a snaží se blíže vysvětlit jejich zájmy a činnost.

název: nakladatel/nakladatelka
skore: 0.88071
Nakladatelé odpovídají za výběr nových materiálů. Rozhodují o tom, které rukopisy poskytnuté knižními redaktory budou zveřejněny. Nakladatelé dohlížejí na produkci, marketing a distribuci těchto textů.

název: oblastní manažer/oblastní manažerka
skore: 0.84843
Oblastní manažeři jsou odpovědní za řízení všech záležitostí souvisejících se společností v určitém zeměpisném regionu nebo pobočce. Dostávají pokyny z ústředí a v závislosti na struktuře společnosti mají za cíl realizovat strategii společnosti a přizpůsobit ji trhu, na němž pobočka působí. Plánují řízení zaměstnanců, komunikaci, marketingovou činnost a sledují výsledky a cíle.

název: nakladatel/nak

### Analýza životopisu s cílovým povoláním
`POST /resume` — nahraje soubor + volitelný target_job
→ `SuggestionResponse` (top_suggestion, target_suggestion)

In [13]:
resume_path = os.path.join('testfiles', 'resume.pdf')
target_job = 'Analytik prodeje' # pokud používáš svůj soubor, můžeš skecifikovat práci pro kterou životopis je

with open(resume_path, 'rb') as f:
    files = {'file': f}
    data = {'target_job': target_job}
    r = requests.post(f'{BASE_URL}/resume', files=files, data=data)

In [ ]:
rprint(r.json())

### Analýza životopisu → práce z různých oblastí pracovního trhu
`POST /resume/domains` — nahraje soubor
→ `DomainResponse` (domains: list of DomainResult, each with domain, occ_count, occupations)

In [19]:
resume_path = os.path.join('testfiles', 'resume.odt')

with open(resume_path, 'rb') as f:
    files = {'file': f}
    r = requests.post(f'{BASE_URL}/resume/domains', files=files)

In [20]:
rprint(r.json())

{'domains': []}

### Dovednosti → povolání dle skore
`POST /text` — JSON s klíčem `skills` (čárkou oddělené dovednosti) + volitelný `target_job`
→ `SkillsResponse` (suggestions: list of Suggestion)

In [21]:
payload = {
    'skills': 'Python, machine learning, analýza dat, SQL, statistika',
    'target_job': None
}
r = requests.post(f'{BASE_URL}/text', json=payload)

In [22]:
rprint(r.json())

{'suggestions': []}

### Skore zadaného povolání
`POST /text/goal` — JSON s `skills` + `target_job`
→ `TargetJobResponse` (top_suggestion, target_suggestion)

In [ ]:
payload = {
    'skills': 'Python, machine learning, analýza dat, SQL, statistika',
    'target_job': 'software inženýr'
}
r = requests.post(f'{BASE_URL}/text/goal', json=payload)

In [ ]:
rprint(r.json())

### Vyhledávání samotných entit
`POST /query` — JSON s `text`, volitelný `entity_type` (all/occupation/skill/skill_group/isco_group), volitelný `n`
→ `List[EntityResult]`

In [ ]:
query = {
    'text': 'programování webových aplikací',
    'entity_type': 'skill',
    'n': 5
}
r = requests.post(f'{BASE_URL}/query', json=query)

In [ ]:
rprint(r.json())